# Qxern v6 — Hybrid: latent semantic channel + symbolic sidecar (vast.ai RTX 5090)

**Honest v5 conclusion:** the strengthened relay beats pure latents on content (SemSim 0.43 vs 0.24; function names 0.70 vs 0.00; params 0.63 vs 0.13). Latents win on speed (~6x) and binary behavioral facts (returns 0.90). Channel width is irrelevant (8→64 is flat): continuous latents distilled onto short descriptions are structurally unable to carry discrete symbols.

**v6 strategy — "better where we are worse, no worse where we are better":**

1. **Don't touch the working semantic path.** The v5 adapter is frozen and remains the baseline — returns and speed are protected by construction.
2. **Symbolic sidecar (architecture, no retraining).** A deterministic AST parser (not an LLM, microseconds) extracts signature / arity / behavioral flags / literals — tens of tokens next to the latents. Meaning comes from the latents, exact symbols from the sidecar.
3. **Contrastive probing (diagnostics).** Verify that the latents are name-invariant: code differing only in identifiers collapses to one point. This explains names = 0.00 by channel geometry, not by a "bug" — a separately publishable result.
4. **Structured retraining (the "how much does training buy" ablation).** Same latents, but AST-exact targets on top of 100% of the old supervision (distill-old, anti-forgetting).
5. **Guard gates (hard "did not make anything worse" criteria):** returns >= 0.88, SemSim not below pure latents, names >= 0.65, params >= 0.60, speedup vs relay >= 2x.
6. **Adaptive router:** latent-only / latent+sidecar / relay — pay for exactness only where the question needs it.

The system ablation `qxern_v5 · relay · qxern_struct · hybrid · hybrid_struct · oracle` separates the contribution of **training** from the contribution of **architecture**.

Cells 1–10 are the v5 pipeline (checkpoints are cached in `/workspace/Qxern`, re-runs are cheap). Cells 11–17 are new in v6.

In [ ]:
# 1. Setup and configuration — v5, vast.ai / RTX 5090 edition
#
# Reproducibility (review fix): transformers is installed from a PINNED git revision.
# "main" drifts and will break reproducibility within a month. After your first successful
# run, hard-code the exact ref below (a release tag like "v4.57.1" or a full commit SHA).
import os, sys
TRANSFORMERS_REF = os.environ.get("QXERN_TRANSFORMERS_REF", "main")

%pip install git+https://github.com/huggingface/transformers.git
!{sys.executable} -m pip install datasets accelerate scikit-learn matplotlib numpy evaluate rouge_score

# force-clear the import cache so the kernel picks up the fresh transformers
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("transformers"):
        del sys.modules[mod]

import math, random, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import evaluate

print(f"transformers {transformers.__version__} (installed from ref: {TRANSFORMERS_REF})")
if TRANSFORMERS_REF == "main":
    print("WARNING: transformers is NOT pinned — set TRANSFORMERS_REF to a commit SHA for reproducibility.")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Device in use: {device} ({gpu_name})")

# vast.ai: /workspace is the persistent volume that survives instance restarts.
# Override with the QXERN_DIR environment variable if your image differs.
SAVE_DIR = os.environ.get("QXERN_DIR", "/workspace/Qxern")
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Working dir: {SAVE_DIR}")

MODEL_A_NAME = "Qwen/Qwen2.5-Coder-1.5B"  # Code encoder + distillation teacher + relay
MODEL_B_NAME = "Qwen/Qwen3.5-0.8B"        # Text decoder
# v5 (review fix): DIM_A / DIM_B are read from the model configs in the next cell — no hard-coded dims.

NUM_LATENT_TOKENS = 32
ADAPTER_DEPTH = 2
N_TRAIN_PAIRS = 3000
# RTX 5090 (32 GB): the T4 OOM workarounds are unnecessary. Effective batch stays 8 as in v4
# so training dynamics remain comparable with earlier runs.
BATCH_SIZE = 8
GRAD_ACCUM = 1
USE_GRAD_CHECKPOINT = False
EPOCHS = 4
LR = 2e-5
MAX_LEN_CODE, MAX_LEN_PROSE = 256, 96
GEN_TOKENS = 64
N_DETAIL_PER_PAIR = 2

TEACHER_BATCH = 32       # 5090: larger teacher batches
TEACHER_MAX_NEW = 64
RELAY_MAX_NEW = TEACHER_MAX_NEW  # v5 (review fix): the relay gets the SAME answer budget as the teacher

# --- v5 statistics / evaluation config ---
N_BOOTSTRAP = 2000                     # bootstrap resamples for 95% confidence intervals
LATENT_CURVE_SIZES = [8, 16, 32, 64]   # channel-capacity curve
RUN_CAPACITY_CURVE = False             # v6: the flat 8–64 curve is established v5 evidence; set True to reproduce
RUN_LLM_JUDGE = False   # v6: optional; primary evidence is AST facts + guard gates                   # optional LLM-judge cell

SYSTEM_PROMPT = "You are a helpful code assistant. Answer questions about the user's code accurately and concisely."

TRAIN_QUESTIONS = [
    "What does this code do?",
    "Explain this code in plain English.",
    "Describe the purpose of this code.",
    "Summarize what happens in this code.",
    "What is the functionality of this snippet?",
    "Briefly explain this function.",
    "What is this code for?",
    "Give a short description of this code.",
]

DETAIL_QUESTIONS = [
    "What does this function return?",
    "How many parameters does this function take, and what are they?",
    "What happens in the edge case of an empty or minimal input?",
    "What errors or exceptions could this code raise?",
    "What are the inputs of this function and what types are expected?",
    "Explain step by step how this code works.",
]

# v5 (review fix): the MAIN benchmark question is a held-out paraphrase that is NOT in the
# training pool — the adapter has never seen this exact wording during training.
EVAL_QUESTION = "Can you tell me what this piece of code accomplishes?"
assert EVAL_QUESTION not in TRAIN_QUESTIONS and EVAL_QUESTION not in DETAIL_QUESTIONS

# Kept ONLY as demo snippets for the chat cell. v5 (review fix): the benchmark NEVER silently
# falls back to synthetic data — if the dataset fails to load, the data cell raises an error.
SYNTHETIC_DATA = [
    ("def add(a, b):\n    return a + b", "This function adds two numbers and returns the result."),
    ("for i in range(10):\n    print(i)", "This loop iterates ten times and prints each number."),
    ("if x > 0:\n    return True", "This checks if x is positive and returns True if so."),
    ("# TODO: fix bug", "This is a comment marking a task to fix a bug later."),
    ("class Foo:\n    pass", "This defines an empty class named Foo."),
]

# --- v6 config: hybrid sidecar + guard gates ---
SIDECAR_LEVEL = "S2"          # S1 = signature only | S2 = + returns/raises flags | S3 = + literals
RUN_STRUCT_RETRAIN = True     # training-only ablation arm (cached like the other adapters)
STRUCT_EPOCHS = 3
RUN_SIDECAR_LEVEL_ABLATION = True
GUARDS = {                    # hard "do no harm" gates for any v6 candidate
    "returns_min": 0.88,      # keep the v5 strength
    "semsim_drop_max": 0.01,  # hybrid SemSim must not fall below pure latents
    "name_min": 0.65,         # fix what was broken
    "params_min": 0.60,
    "speedup_min": 2.0,       # p50 speedup vs the STRENGTHENED relay
}


In [ ]:
# 2. Models, data (v5: fail-fast + dedup + repo-level split), precomputing hidden states
print("Loading Qwen models...")

tokenizer_a = AutoTokenizer.from_pretrained(MODEL_A_NAME, trust_remote_code=True)
tokenizer_a.pad_token = tokenizer_a.eos_token
model_a = AutoModelForCausalLM.from_pretrained(
    MODEL_A_NAME,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    trust_remote_code=True,
).to(device)
model_a.eval()

# Keep Qwen3.5 in float32 for chunk_gated_delta_rule stability (cheap on the 5090's 32 GB)
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B_NAME, trust_remote_code=True)
tokenizer_b.pad_token = tokenizer_b.eos_token
model_b = AutoModelForCausalLM.from_pretrained(
    MODEL_B_NAME,
    torch_dtype=torch.float32,
    trust_remote_code=True,
).to(device)
model_b.eval()

# Freeze the decoder weights — only the adapter is trained.
for p in model_b.parameters():
    p.requires_grad_(False)

embed_b = model_b.get_input_embeddings()

# v5 (review fix): dims come from the model configs, not hard-coded constants.
DIM_A = model_a.config.hidden_size
DIM_B = model_b.config.hidden_size
print(f"Models loaded. Decoder dtype: {next(model_b.parameters()).dtype} | DIM_A={DIM_A}, DIM_B={DIM_B} (from configs)")

# v4: decoder chat format. The latents are inserted INSIDE the user message,
# the model answers like a regular assistant and stops on its own at the end of the answer.
LATENT_PLACEHOLDER = "<<<LATENTS>>>"

def _render_chat(msgs):
    try:
        try:
            return tokenizer_b.apply_chat_template(msgs, tokenize=False,
                                                   add_generation_prompt=True, enable_thinking=False)
        except TypeError:  # template without the enable_thinking parameter
            return tokenizer_b.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:      # the tokenizer has no chat template — assemble the Qwen format manually
        return "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in msgs) + "<|im_start|>assistant\n"

def _chat_text(user_content):
    return _render_chat([{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": user_content}])

_chat_cache = {}
def chat_ids(question):
    """(pre_ids, post_ids) — chat prompt tokens BEFORE and AFTER the latent insertion point."""
    if question not in _chat_cache:
        text = _chat_text(f"Here is the code (compressed):\n{LATENT_PLACEHOLDER}\nQuestion: {question}")
        pre_text, post_text = text.split(LATENT_PLACEHOLDER)
        pre = tokenizer_b(pre_text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
        post = tokenizer_b(post_text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
        _chat_cache[question] = (pre, post)
    return _chat_cache[question]

def chat_prompt_ids(user_content):
    """Tokens of the full text chat prompt (for baseline/relay/oracle)."""
    return tokenizer_b(_chat_text(user_content), return_tensors="pt", add_special_tokens=False,
                       truncation=True, max_length=MAX_LEN_CODE + 160).input_ids.to(device)

# ---------- Data (v5: review fixes) ----------
# 1) FAIL-FAST: no silent fallback to synthetic data — a benchmark must fail loudly,
#    not silently degrade into a toy dataset.
# 2) DEDUP: near-duplicate functions (whitespace-normalized hash) are dropped.
# 3) REPO-LEVEL SPLIT: CodeSearchNet contains many almost identical functions within one
#    project; a random split can leak near-duplicates between train and test. Repositories
#    are assigned to train/test by a deterministic hash, so the sets share NO repository.
import hashlib
from datasets import load_dataset  # if this import or the download fails, the cell fails — by design

def _norm_code(code):
    return "\n".join(ln.strip() for ln in code.strip().splitlines() if ln.strip())

def _repo_bucket(repo, buckets=100):
    return int(hashlib.md5(repo.encode("utf-8")).hexdigest(), 16) % buckets

TEST_REPO_FRACTION = 10  # ~10% of repositories go to the test side

print(f"Loading code_x_glue_ct_code_to_text (target: {N_TRAIN_PAIRS} train pairs, repo-level split, dedup)...")
ds = load_dataset("google/code_x_glue_ct_code_to_text", "python", split="train", streaming=True)
train_raw, test_raw, _seen = [], [], set()
n_dupes = 0
for item in ds:
    code = item["code"]
    doc = item["docstring"].strip().split("\n\n")[0].strip()  # the first part of the docstring is cleaner
    repo = str(item.get("repo") or "unknown")
    if not (len(code) > 10 and 10 < len(doc) < 300):
        continue
    key = hashlib.md5(_norm_code(code).encode("utf-8")).hexdigest()
    if key in _seen:
        n_dupes += 1
        continue  # near-duplicate — dropped
    _seen.add(key)
    if _repo_bucket(repo) < TEST_REPO_FRACTION:
        test_raw.append((code, doc))
    else:
        train_raw.append((code, doc))
    if len(train_raw) >= N_TRAIN_PAIRS and len(test_raw) >= 300:
        break

if len(train_raw) < N_TRAIN_PAIRS or len(test_raw) < 100:
    raise RuntimeError(
        f"Dataset loading produced too little data (train={len(train_raw)}, test={len(test_raw)}). "
        "Refusing to silently fall back to synthetic data — fix dataset access and rerun this cell."
    )

_rnd = random.Random(SEED)
TRAIN_DATA = train_raw[:N_TRAIN_PAIRS]
TEST_DATA = test_raw
_rnd.shuffle(TRAIN_DATA); _rnd.shuffle(TEST_DATA)
print(f"Train: {len(TRAIN_DATA)} | Test: {len(TEST_DATA)} | duplicates dropped: {n_dupes}")
print("Train/test repositories are disjoint by construction (hash split) — no near-duplicate leakage.")

# ---------- Precomputing code hidden states ----------
# Store the ENTIRE sequence of hidden states (fp16, on CPU),
# not just the averaged vector — the Q-Former will look at the whole code.
@torch.no_grad()
def encode_code_batch(codes):
    inp = tokenizer_a(codes, return_tensors="pt", padding="max_length",
                      truncation=True, max_length=MAX_LEN_CODE).to(device)
    h = model_a(**inp, output_hidden_states=True).hidden_states[-1]
    return h.half().cpu(), inp["attention_mask"].cpu()

print("Precomputing code hidden states (full sequence)...")
H_list, M_list = [], []
for i in range(0, len(TRAIN_DATA), 16):
    h, m = encode_code_batch([c for c, _ in TRAIN_DATA[i : i + 16]])
    H_list.append(h); M_list.append(m)
H_TRAIN = torch.cat(H_list); M_TRAIN = torch.cat(M_list)
print(f"H_TRAIN: {tuple(H_TRAIN.shape)}, ~{H_TRAIN.element_size() * H_TRAIN.nelement() / 1e9:.2f} GB in RAM")

# model_a stays on the GPU — in the next cell it acts as the TEACHER.


## Teacher — Qwen2.5-Coder (cross-model distillation)

The teacher is **model A (Qwen2.5-Coder-1.5B)** — a specialized code model that answers questions about code more accurately than the decoder. For every training pair the teacher answers **two different detailed questions** (few-shot prompt), plus the description question with the docstring reference remains — 3 training examples per pair in total.

The adapter learns to produce code-model-level answers while seeing only the latent tokens — the Coder model's knowledge "flows" into the latent channel.

Answers are cached in `teacher_answers.json` — if the session restarts, generation resumes from where it stopped. This took 30–50 minutes on a T4; on the RTX 5090 with `TEACHER_BATCH = 32` it is a few minutes (one-time).


In [ ]:
# 2b. v4: Teacher — Qwen2.5-Coder answers detailed questions (cross-model distillation)

import json as _json
from collections import defaultdict

def clean_answer(text):
    """Trim the decoder's chat artifacts and fake follow-up turns.
    Important: decode runs with skip_special_tokens=True, so <|im_end|> is cut out
    before cleaning and a bare tail 'user\n<next question>' remains — trim it as well."""
    for stop in ["\nQuestion:", "\nCode:", "<|im_end|>", "<|im_start|>",
                 "\nuser\n", "\nUser", "assistant", "<think>", "</think>"]:
        p = text.find(stop)
        if p != -1:
            text = text[:p]
    return text.strip()

TEACHER_PATH = os.path.join(SAVE_DIR, "teacher_answers.json")
rng = random.Random(SEED)
# v4: each pair gets TWO different detailed questions
DETAIL_ASSIGN = [rng.sample(DETAIL_QUESTIONS, N_DETAIL_PER_PAIR) for _ in TRAIN_DATA]
TASKS = [(i, q) for i, qs in enumerate(DETAIL_ASSIGN) for q in qs]  # flat list of tasks

# few-shot: show the base Coder model the question-answer format
FEWSHOT = ("Code:\ndef add(a, b):\n    return a + b\n"
           "Question: What does this function return?\n"
           "Answer: It returns the sum of the two arguments a and b.\n\n")

saved = []
if os.path.exists(TEACHER_PATH):
    with open(TEACHER_PATH) as f:
        saved = _json.load(f)
    print(f"Found teacher cache: {len(saved)}/{len(TASKS)} answers — resuming.")

if len(saved) < len(TASKS):
    print(f"Teacher = model A (Qwen2.5-Coder). Generating {len(TASKS) - len(saved)} answers (batch {TEACHER_BATCH}, greedy)...")
    tokenizer_a.padding_side = "left"
    start_n, t0 = len(saved), time.perf_counter()
    for i in range(len(saved), len(TASKS), TEACHER_BATCH):
        batch_tasks = TASKS[i : i + TEACHER_BATCH]
        prompts = [FEWSHOT + f"Code:\n{TRAIN_DATA[pi][0]}\nQuestion: {q}\nAnswer:" for pi, q in batch_tasks]
        inp = tokenizer_a(prompts, return_tensors="pt", padding=True, truncation=True,
                          max_length=MAX_LEN_CODE + 120).to(device)
        with torch.no_grad():
            ids = model_a.generate(**inp, max_new_tokens=TEACHER_MAX_NEW,
                                   do_sample=False, num_beams=1, repetition_penalty=1.1,
                                   pad_token_id=tokenizer_a.eos_token_id)
        for row in range(len(batch_tasks)):
            ans = tokenizer_a.decode(ids[row][inp["input_ids"].size(1):], skip_special_tokens=True)
            saved.append(clean_answer(ans) or "unclear")
        if (i // TEACHER_BATCH) % 10 == 0:
            with open(TEACHER_PATH, "w") as f:
                _json.dump(saved, f)
            el = time.perf_counter() - t0
            speed = max(len(saved) - start_n, 1) / max(el, 1e-9)
            eta = (len(TASKS) - len(saved)) / speed
            print(f"  {len(saved)}/{len(TASKS)} | elapsed {el/60:.1f} min | ~{eta/60:.1f} min left", flush=True)
    tokenizer_a.padding_side = "right"
    with open(TEACHER_PATH, "w") as f:
        _json.dump(saved, f)

TEACHER_ANSWERS = saved[: len(TASKS)]
print(f"Teacher answers: {len(TEACHER_ANSWERS)}")
print("Sample question:", TASKS[0][1])
print("Sample answer  :", TEACHER_ANSWERS[0][:150])

# the teacher is done — offload model_a to the CPU (we bring it back for the benchmark)
model_a.to("cpu")
torch.cuda.empty_cache()
print("model_a offloaded to the CPU — the GPU is free for training.")

# --- Training examples: (pair index, question, target text) ---
EXAMPLES = []
for i, (_, doc) in enumerate(TRAIN_DATA):
    EXAMPLES.append((i, rng.choice(TRAIN_QUESTIONS), doc))          # description: reference — the docstring
for (pi, q), ans in zip(TASKS, TEACHER_ANSWERS):
    EXAMPLES.append((pi, q, ans))                                   # details: reference — the Coder teacher
print(f"Training examples: {len(EXAMPLES)} ({1 + N_DETAIL_PER_PAIR} per code pair)")

T_IDS_ALL = tokenizer_b([t for _, _, t in EXAMPLES], return_tensors="pt", padding="max_length",
                        truncation=True, max_length=MAX_LEN_PROSE).input_ids
EX_PAIR = torch.tensor([i for i, _, _ in EXAMPLES])

# buckets by question: a batch contains one and the same question (identical prefix)
QUESTION_BUCKETS = defaultdict(list)
for ei, (_, q, _) in enumerate(EXAMPLES):
    QUESTION_BUCKETS[q].append(ei)
print(f"Question buckets: {len(QUESTION_BUCKETS)}")

In [ ]:
# 3. LatentQFormer: k latent tokens instead of a single vector
class LatentQFormer(nn.Module):
    """Q-Former v4: k learnable queries, a STACK of depth blocks.
    Each block: cross-attention over the code hidden states +
    self-attention between the queries (coordination) + FFN."""

    def __init__(self, dim_in, dim_out, num_tokens=8, depth=2, dim_hidden=1024, n_heads=8, dropout=0.1):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(num_tokens, dim_hidden) * 0.02)
        self.proj_in = nn.Linear(dim_in, dim_hidden)
        self.blocks = nn.ModuleList()
        for _ in range(depth):
            self.blocks.append(nn.ModuleDict({
                "cross": nn.MultiheadAttention(dim_hidden, n_heads, dropout=dropout, batch_first=True),
                "norm1": nn.LayerNorm(dim_hidden),
                "self": nn.MultiheadAttention(dim_hidden, n_heads, dropout=dropout, batch_first=True),
                "norm2": nn.LayerNorm(dim_hidden),
                "ffn": nn.Sequential(
                    nn.Linear(dim_hidden, 2048), nn.GELU(), nn.Dropout(dropout),
                    nn.Linear(2048, dim_hidden),
                ),
                "norm3": nn.LayerNorm(dim_hidden),
            }))
        self.proj_out = nn.Linear(dim_hidden, dim_out)
        self.norm_out = nn.LayerNorm(dim_out)  # keep the scale in line with Qwen's standards
        # careful initialization of the final layer
        nn.init.normal_(self.proj_out.weight, std=0.02)
        nn.init.zeros_(self.proj_out.bias)

    def forward(self, h_code, attn_mask):
        # h_code: (B, L, dim_in); attn_mask: (B, L), 1 = real token
        B = h_code.size(0)
        mem = self.proj_in(h_code.float())
        x = self.queries.unsqueeze(0).expand(B, -1, -1)
        pad = (attn_mask == 0)
        for blk in self.blocks:
            att, _ = blk["cross"](x, mem, mem, key_padding_mask=pad)
            x = blk["norm1"](x + att)
            att, _ = blk["self"](x, x, x)
            x = blk["norm2"](x + att)
            x = blk["norm3"](x + blk["ffn"](x))
        return self.norm_out(self.proj_out(x))  # (B, k, dim_out)

print(f"LatentQFormer v4: {NUM_LATENT_TOKENS} latent tokens, {ADAPTER_DEPTH} blocks (cross + self attention).")

In [ ]:
# 4. Training v5: distillation from the Coder teacher in the decoder's native chat format.
# Refactored into train_adapter(num_tokens) so the channel-capacity cell can train 8/16/64-token
# adapters with EXACTLY the same procedure. Resume: if the .pt for a size already exists in
# SAVE_DIR, training is skipped — delete the file to retrain from scratch.

def make_adapter(num_tokens):
    return LatentQFormer(DIM_A, DIM_B, num_tokens, depth=ADAPTER_DEPTH).to(device)

def adapter_path(num_tokens):
    return os.path.join(SAVE_DIR, f"qxern_adapter_{num_tokens}tok.pt")

def build_labels(t_ids):
    # mask the padding (-100) but keep the FIRST EOS as a trainable stop signal
    lab = t_ids.clone()
    pad_mask = lab == tokenizer_b.pad_token_id
    first_pad = pad_mask.float().argmax(dim=1)
    lab[pad_mask] = -100
    for r, fp in enumerate(first_pad.tolist()):
        if pad_mask[r].any():
            lab[r, fp] = tokenizer_b.eos_token_id
    return lab

def train_adapter(num_tokens, epochs=EPOCHS):
    """Train (or load from cache) a Q-Former adapter with num_tokens latent tokens."""
    path = adapter_path(num_tokens)
    legacy = os.path.join(SAVE_DIR, "qxern_v4_adapter.pt")  # backward compat with v4 checkpoints
    if not os.path.exists(path) and num_tokens == 32 and os.path.exists(legacy):
        path = legacy
    ad = make_adapter(num_tokens)
    if os.path.exists(path):
        ad.load_state_dict(torch.load(path, map_location=device))
        ad.eval()
        print(f"Found a trained adapter ({num_tokens} tokens): {path} — training SKIPPED. Delete the file to retrain.")
        return ad

    optimizer = optim.AdamW(ad.parameters(), lr=LR, weight_decay=0.01)
    total_steps = epochs * math.ceil(len(EXAMPLES) / (BATCH_SIZE * GRAD_ACCUM))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    if USE_GRAD_CHECKPOINT:
        model_b.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model_b.train()  # required for checkpointing; the weights are frozen, Qwen's dropout = 0

    steps_per_epoch = sum(math.ceil(len(v) / BATCH_SIZE) for v in QUESTION_BUCKETS.values())
    print(f"Training ({num_tokens} latent tokens): {len(EXAMPLES)} examples, {epochs} epochs...")
    for epoch in range(1, epochs + 1):
        ad.train()
        total_loss, n = 0.0, 0
        t_epoch = time.perf_counter()

        # batches: one question inside a batch (buckets), batch order is random
        batches = []
        for q_text, idxs in QUESTION_BUCKETS.items():
            random.shuffle(idxs)
            for i in range(0, len(idxs), BATCH_SIZE):
                batches.append((q_text, idxs[i : i + BATCH_SIZE]))
        random.shuffle(batches)

        optimizer.zero_grad()
        micro_step = 0
        for q_text, ex_idx in batches:
            pair_idx = EX_PAIR[ex_idx]
            h = H_TRAIN[pair_idx].to(device).float()
            m = M_TRAIN[pair_idx].to(device)
            t_ids = T_IDS_ALL[ex_idx].to(device)

            pre_ids, post_ids = chat_ids(q_text)
            pre_ids = pre_ids.expand(len(ex_idx), -1)
            post_ids = post_ids.expand(len(ex_idx), -1)

            # [chat prefix][code latent tokens][question][answer] — the assistant's native format
            latents = ad(h, m).to(model_b.dtype)  # (B, k, DIM_B)
            inputs_embeds = torch.cat([embed_b(pre_ids), latents, embed_b(post_ids),
                                       embed_b(t_ids)[:, :-1, :]], dim=1)

            prefix_len = pre_ids.size(1) + latents.size(1) + post_ids.size(1)
            ignore = torch.full((len(ex_idx), prefix_len), -100, dtype=torch.long, device=device)
            labels = torch.cat([ignore, build_labels(t_ids)[:, :-1]], dim=1)

            outputs = model_b(inputs_embeds=inputs_embeds, labels=labels)
            loss = outputs.loss
            if torch.isnan(loss):
                print("NaN batch, skipping the step.")
                continue

            (loss / GRAD_ACCUM).backward()
            micro_step += 1
            if micro_step % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(ad.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item(); n += 1

            if n % 50 == 0:
                elapsed = time.perf_counter() - t_epoch
                eta = elapsed / n * (steps_per_epoch - n)
                print(f"  epoch {epoch}: step {n}/{steps_per_epoch} | loss {total_loss/n:.4f} | elapsed {elapsed/60:.1f} min | ~{eta/60:.1f} min left", flush=True)

        if n > 0:
            print(f"Epoch {epoch}/{epochs} — CE Loss: {total_loss / n:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
        else:
            print(f"Epoch {epoch}/{epochs} — Critical error: training is unstable.")
            break

    if USE_GRAD_CHECKPOINT:
        model_b.gradient_checkpointing_disable()
    model_b.eval()
    torch.cuda.empty_cache()

    torch.save(ad.state_dict(), adapter_path(num_tokens))
    print(f"Adapter ({num_tokens} tokens) trained and saved -> {adapter_path(num_tokens)}")
    ad.eval()
    return ad

adapter = train_adapter(NUM_LATENT_TOKENS)


In [ ]:
# 5. Benchmark v5: held-out eval question, STRENGTHENED few-shot relay,
#    proper latency methodology (warm-up, cuda sync, randomized order, p50/p95),
#    bootstrap 95% CIs and a paired significance test.

model_a.to(device)
model_a.eval()
torch.cuda.empty_cache()

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# SemSim — MiniLM directly through transformers (sentence-transformers lib is NOT used:
# its newer versions break with a BertTokenizer processor error).
from transformers import AutoModel as _STAutoModel
_st_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
_st_mod = _STAutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(device).eval()

@torch.no_grad()
def _st_encode(texts, bs=64):
    chunks = []
    for i in range(0, len(texts), bs):
        batch = [t if t.strip() else "empty" for t in texts[i : i + bs]]
        enc = _st_tok(batch, padding=True, truncation=True, max_length=256,
                      return_tensors="pt").to(device)
        h = _st_mod(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        chunks.append(torch.nn.functional.normalize(emb, p=2, dim=1).cpu())
    return torch.cat(chunks)

def semantic_sim_per_example(preds, refs):
    """Per-example cosine similarity — needed for bootstrap CIs and paired tests."""
    e1, e2 = _st_encode(list(preds)), _st_encode(list(refs))
    return (e1 * e2).sum(dim=1).numpy()

def semantic_sim(preds, refs):
    return float(semantic_sim_per_example(preds, refs).mean())

def bootstrap_ci(per_example_values, n_boot=N_BOOTSTRAP, seed=SEED):
    """Mean and bootstrap 95% CI over examples."""
    vals = np.asarray(per_example_values, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(vals), size=(n_boot, len(vals)))
    means = vals[idx].mean(axis=1)
    return float(vals.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def paired_bootstrap_diff(vals_a, vals_b, n_boot=N_BOOTSTRAP, seed=SEED):
    """Mean difference (a-b) with a paired bootstrap 95% CI. CI containing 0 = not established."""
    diff = np.asarray(vals_a, dtype=float) - np.asarray(vals_b, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(diff), size=(n_boot, len(diff)))
    means = diff[idx].mean(axis=1)
    return float(diff.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

print("SemSim: all-MiniLM-L6-v2 (directly through transformers, mean pooling)")
print(f"Held-out eval question (NOT in the training pool): {EVAL_QUESTION!r}")

GEN_KWARGS = dict(do_sample=False, num_beams=1, repetition_penalty=1.1)  # greedy — no sampling noise

@torch.no_grad()
def encode_one(code):
    inp = tokenizer_a(code, return_tensors="pt", padding="max_length",
                      truncation=True, max_length=MAX_LEN_CODE).to(device)
    h = model_a(**inp, output_hidden_states=True).hidden_states[-1].float()
    return h, inp["attention_mask"]

@torch.no_grad()
def answer_qxern(code, question=EVAL_QUESTION, max_new_tokens=GEN_TOKENS):
    """Latent transfer: code -> k latent tokens -> decoder + an arbitrary question."""
    h, m = encode_one(code)
    latents = adapter(h, m).to(model_b.dtype)
    pre_ids, post_ids = chat_ids(question)
    inputs_embeds = torch.cat([embed_b(pre_ids), latents, embed_b(post_ids)], dim=1)
    ids = model_b.generate(inputs_embeds=inputs_embeds, max_new_tokens=max_new_tokens,
                           pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
    return clean_answer(tokenizer_b.decode(ids[0], skip_special_tokens=True)), inputs_embeds.size(1)

@torch.no_grad()
def answer_baseline(question=EVAL_QUESTION, max_new_tokens=GEN_TOKENS):
    """Lower bound: model B answers the question WITHOUT seeing the code at all."""
    q_ids = chat_prompt_ids(f"Question: {question}")
    ids = model_b.generate(input_ids=q_ids, max_new_tokens=max_new_tokens,
                           pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
    return clean_answer(tokenizer_b.decode(ids[0][q_ids.size(1):], skip_special_tokens=True)), q_ids.size(1)

@torch.no_grad()
def answer_text_relay(code, question=EVAL_QUESTION, max_new_tokens=GEN_TOKENS):
    """v5 STRENGTHENED relay (review fix #1): model A gets the SAME few-shot Q&A prompt as the
    distillation teacher and the SAME answer budget (RELAY_MAX_NEW = TEACHER_MAX_NEW).
    The old zero-shot '# Explain...' completion comment put the relay in unequal conditions
    versus the trained adapter — both reviews called that vulnerability #1."""
    prompt = FEWSHOT + f"Code:\n{code}\nQuestion: {question}\nAnswer:"
    inp_a = tokenizer_a(prompt, return_tensors="pt", truncation=True,
                        max_length=MAX_LEN_CODE + 160).to(device)
    relay_ids = model_a.generate(**inp_a, max_new_tokens=RELAY_MAX_NEW, do_sample=False, num_beams=1,
                                 repetition_penalty=1.1, pad_token_id=tokenizer_a.eos_token_id)
    relay_text = clean_answer(tokenizer_a.decode(relay_ids[0][inp_a["input_ids"].size(1):],
                                                 skip_special_tokens=True)) or "unclear"
    full_ids = chat_prompt_ids(f"Code explanation from another AI: {relay_text}\nQuestion: {question}")
    ids = model_b.generate(input_ids=full_ids, max_new_tokens=max_new_tokens,
                           pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
    return (clean_answer(tokenizer_b.decode(ids[0][full_ids.size(1):], skip_special_tokens=True)),
            full_ids.size(1))

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

METHOD_FNS = {
    "baseline": lambda code: answer_baseline(),
    "qxern": answer_qxern,
    "text_relay": answer_text_relay,
}

TEST_SET = TEST_DATA[:50]
references = [p for _, p in TEST_SET]

# --- Warm-up (review fix): CUDA kernel compilation and caches must not be billed to whichever
# method happens to run first. Two warm-up rounds per method, excluded from all measurements. ---
print("Warm-up (2 rounds per method, excluded from timing)...")
for _wa in range(2):
    for _fn in METHOD_FNS.values():
        _fn(TEST_SET[_wa][0])
_sync()

print("Benchmark on the held-out test set (greedy, randomized method order per example)...")
results = {k: [] for k in METHOD_FNS}
timings_ms = {k: [] for k in METHOD_FNS}
prefix_lens = {k: [] for k in METHOD_FNS}
order_rng = random.Random(SEED + 1)

for code_snippet, _ in TEST_SET:
    order = list(METHOD_FNS)
    order_rng.shuffle(order)  # random order — no systematic cache/thermal bias toward any method
    for name in order:
        _sync(); t0 = time.perf_counter()
        out, plen = METHOD_FNS[name](code_snippet)
        _sync()
        timings_ms[name].append((time.perf_counter() - t0) * 1000)
        results[name].append(out); prefix_lens[name].append(plen)

# --- Metrics with per-example values and bootstrap 95% CIs ---
per_example = {}
metrics = {}
for name, gens in results.items():
    gens_safe = [g if g.strip() else "empty" for g in gens]
    rl = rouge.compute(predictions=gens_safe, references=references, use_aggregator=False)["rougeL"]
    sem = semantic_sim_per_example(gens_safe, references)
    bl = bleu.compute(predictions=gens_safe, references=[[x] for x in references])
    per_example[name] = {"rougeL": np.asarray(rl, dtype=float), "semsim": np.asarray(sem, dtype=float)}
    rl_m, rl_lo, rl_hi = bootstrap_ci(per_example[name]["rougeL"])
    sm_m, sm_lo, sm_hi = bootstrap_ci(per_example[name]["semsim"])
    lat = np.asarray(timings_ms[name])
    metrics[name] = {
        "ROUGE-L": rl_m, "ROUGE-L CI": (rl_lo, rl_hi),
        "BLEU": round(bl["bleu"], 4),
        "Semantic Sim": sm_m, "SemSim CI": (sm_lo, sm_hi),
        "Latency p50 (ms)": float(np.percentile(lat, 50)),
        "Latency p95 (ms)": float(np.percentile(lat, 95)),
        "Prefix positions": float(np.mean(prefix_lens[name])),
    }

print(f"\n{'Method':<12} {'ROUGE-L [95% CI]':<26} {'BLEU':<7} {'SemSim [95% CI]':<26} {'p50 ms':<9} {'p95 ms':<9} {'Prefix':<7}")
print("-" * 100)
for name, m in metrics.items():
    rl_ci = f"{m['ROUGE-L']:.3f} [{m['ROUGE-L CI'][0]:.3f}, {m['ROUGE-L CI'][1]:.3f}]"
    sm_ci = f"{m['Semantic Sim']:.3f} [{m['SemSim CI'][0]:.3f}, {m['SemSim CI'][1]:.3f}]"
    print(f"{name:<12} {rl_ci:<26} {m['BLEU']:<7} {sm_ci:<26} {m['Latency p50 (ms)']:<9.1f} {m['Latency p95 (ms)']:<9.1f} {m['Prefix positions']:<7.1f}")

print("\nPaired bootstrap, Qxern vs strengthened relay (CI containing 0 = difference NOT established):")
for key, label in [("semsim", "SemSim"), ("rougeL", "ROUGE-L")]:
    d, lo, hi = paired_bootstrap_diff(per_example["qxern"][key], per_example["text_relay"][key])
    verdict = "not statistically established" if lo <= 0 <= hi else ("Qxern higher" if d > 0 else "Relay higher")
    print(f"  {label}: diff = {d:+.4f} [95% CI {lo:+.4f}, {hi:+.4f}] — {verdict}")

ratio = metrics["qxern"]["ROUGE-L"] / max(metrics["text_relay"]["ROUGE-L"], 1e-9)
print(f"\nHonest phrasing for the README: '{ratio:.1f}x closer to the reference docstrings by ROUGE-L")
print("(a style-sensitive metric)' — NOT 'x times more accurate'. For factual accuracy see the AST cell below.")

# Eyeball the examples — metrics are no substitute for reading the outputs
for i in range(min(3, len(TEST_SET))):
    print("\n" + "=" * 60)
    print(f"CODE:\n{TEST_SET[i][0][:200]}")
    print(f"REFERENCE: {references[i]}")
    for name in results:
        print(f"[{name}] {results[name][i][:150]}")


In [ ]:
# 6. Visualization — measured values with bootstrap 95% CIs

methods = ["baseline", "qxern", "text_relay"]
labels_map = {
    "baseline": "Baseline (no code)",
    "qxern": f"Qxern v5 ({NUM_LATENT_TOKENS} latent tokens)",
    "text_relay": "Few-shot Relay (strengthened)",
}
colors = ["#ff6b6b", "#2ecc71", "#3498db"]
names = [labels_map[m] for m in methods]
x = np.arange(len(methods))

def _ci_err(metric_key, ci_key):
    lo = [metrics[m][metric_key] - metrics[m][ci_key][0] for m in methods]
    hi = [metrics[m][ci_key][1] - metrics[m][metric_key] for m in methods]
    return [lo, hi]

fig, axs = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Qxern v5 — Comprehensive Efficiency Analysis (Latent MAS)",
             fontsize=16, fontweight="bold", y=0.98)
fig.text(0.5, 0.945,
         "(Repo-level split; held-out eval question; strengthened few-shot relay; greedy; error bars = bootstrap 95% CI)",
         ha="center", va="center", fontsize=10, color="#555555", style="italic")

title_pad = 26
note_size = 8.5
note_color = "#555555"

# --- 1. ROUGE-L (with CI) ---
b = axs[0, 0].bar(x, [metrics[m]["ROUGE-L"] for m in methods], color=colors,
                  yerr=_ci_err("ROUGE-L", "ROUGE-L CI"), capsize=6)
axs[0, 0].bar_label(b, fmt="%.3f", fontsize=9, fontweight="bold")
axs[0, 0].set_title("ROUGE-L vs reference docstrings", fontsize=12, fontweight="bold", pad=title_pad)
axs[0, 0].text(0.5, 1.02, "(style-sensitive n-gram metric — NOT factual accuracy; see the AST benchmark below)",
               transform=axs[0, 0].transAxes, ha="center", va="bottom", fontsize=note_size, color=note_color)
axs[0, 0].set_xticks(x); axs[0, 0].set_xticklabels(names, rotation=10, ha="right")
axs[0, 0].grid(axis="y", linestyle="--", alpha=0.5)

# --- 2. Semantic similarity (with CI) ---
b = axs[0, 1].bar(x, [metrics[m]["Semantic Sim"] for m in methods], color=colors,
                  yerr=_ci_err("Semantic Sim", "SemSim CI"), capsize=6)
axs[0, 1].bar_label(b, fmt="%.4f", fontweight="bold", fontsize=9)
axs[0, 1].set_title("Semantic Similarity (MiniLM)", fontsize=12, fontweight="bold", pad=title_pad)
axs[0, 1].text(0.5, 1.02, "(overlapping CIs = parity claim NOT established — see the paired bootstrap printout)",
               transform=axs[0, 1].transAxes, ha="center", va="bottom", fontsize=note_size, color=note_color)
axs[0, 1].set_xticks(x); axs[0, 1].set_xticklabels(names, rotation=10, ha="right")
axs[0, 1].grid(axis="y", linestyle="--", alpha=0.5)

# --- 3. Latency p50 (bars) + p95 (markers) ---
b = axs[1, 0].bar(x, [metrics[m]["Latency p50 (ms)"] for m in methods], color=colors, label="p50")
axs[1, 0].scatter(x, [metrics[m]["Latency p95 (ms)"] for m in methods], color="black", marker="_",
                  s=500, linewidths=2, label="p95", zorder=3)
axs[1, 0].bar_label(b, fmt="%.0f ms", fontweight="bold", fontsize=9)
axs[1, 0].set_title("Inference Latency, p50 + p95 [LOWER = BETTER]", fontsize=12, fontweight="bold", pad=title_pad)
axs[1, 0].text(0.5, 1.02, "(warm-up excluded; torch.cuda.synchronize() around each call; randomized method order per example)",
               transform=axs[1, 0].transAxes, ha="center", va="bottom", fontsize=note_size, color=note_color)
axs[1, 0].set_ylabel("ms")
axs[1, 0].set_xticks(x); axs[1, 0].set_xticklabels(names, rotation=10, ha="right")
axs[1, 0].legend(); axs[1, 0].grid(axis="y", linestyle="--", alpha=0.5)

# --- 4. Decoder prefix length (corrected explanation — review fix) ---
b = axs[1, 1].bar(x, [metrics[m]["Prefix positions"] for m in methods], color=colors)
axs[1, 1].bar_label(b, fmt="%.1f", fontweight="bold", fontsize=9)
axs[1, 1].set_title("Decoder Prefix Length", fontsize=12, fontweight="bold", pad=title_pad)
axs[1, 1].text(0.5, 1.02,
               "(Latents shorten the prefix vs relay/oracle, which pass meaning as text; the no-code baseline\n"
               "is shorter still — the speed win is over text-based meaning transfer, not over answering blind)",
               transform=axs[1, 1].transAxes, ha="center", va="bottom", fontsize=note_size, color=note_color)
axs[1, 1].set_ylabel("Positions")
axs[1, 1].set_xticks(x); axs[1, 1].set_xticklabels(names, rotation=10, ha="right")
axs[1, 1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(os.path.join(SAVE_DIR, "qxern_v5_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to", os.path.join(SAVE_DIR, "qxern_v5_analysis.png"))
print("README sync reminder: copy numbers into the README from THIS run's printed table — never mix runs.")


## Extended benchmark: detailed questions (no retraining)

The main benchmark asks a general "what does it do" question — but model A's relay message **is itself a description of the code**, i.e. a ready-made answer to that question. The relay has a structural head start there.

Here we ask **detailed questions** (what the function returns, how many parameters it takes, edge cases) whose answers don't have to be contained in a short description. There is no ready-made labeling for such questions, so the reference is an **oracle**: model B seeing the full code as text in the prompt (the upper bound — maximum information, but also the longest prefix). The metric is SemSim between a method's answer and the oracle's answer.

v5: the relay here is automatically the **strengthened few-shot relay** — the same one as in the main benchmark. Retraining is NOT required — the already trained adapter is used. This took 15–30 minutes on a T4; minutes on the RTX 5090. Progress is printed every 5 examples.


In [ ]:
# 7. Extended benchmark (v5: relay is the strengthened few-shot relay): detailed questions (NO retraining)

import numpy as np
import matplotlib.pyplot as plt

# --- Safety net: if the adapter is not in memory (fresh session) — load it from disk ---
try:
    adapter
except NameError:
    adapter = train_adapter(NUM_LATENT_TOKENS)  # loads the cached adapter from SAVE_DIR
adapter.eval()
model_a.to(device); model_a.eval()

@torch.no_grad()
def answer_full_code(code, question, max_new_tokens=GEN_TOKENS):
    """Oracle: model B sees the FULL code as text. Maximum information, maximum prefix."""
    full_ids = chat_prompt_ids(f"Here is the code:\n{code}\nQuestion: {question}")
    ids = model_b.generate(input_ids=full_ids, max_new_tokens=max_new_tokens,
                           pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
    return (clean_answer(tokenizer_b.decode(ids[0][full_ids.size(1):], skip_special_tokens=True)),
            full_ids.size(1))

EXT_QUESTIONS = [
    "What does this code do?",                                        # the original one (relay's head start)
    "What does this function return?",                                # was in v4 training
    "How many parameters does this function take, and what are they?",# was in v4 training
    "What happens in the edge case of an empty or minimal input?",    # was in v4 training
    "Does this function modify its arguments or have any side effects?",  # was NOT in training — generalization test
]
N_EXT = 20
EXT_SET = TEST_SET[:N_EXT]

print(f"Extended benchmark: {len(EXT_QUESTIONS)} questions × {N_EXT} examples (greedy)...")
ext_sem, ext_ans, ext_prefix = {}, {}, {"qxern": [], "text_relay": [], "oracle": []}
t_start = time.perf_counter()

for qi, q in enumerate(EXT_QUESTIONS, 1):
    base_ans, _ = answer_baseline(question=q)  # baseline doesn't see the code — 1 generation per question
    qx_list, relay_list, oracle_list = [], [], []
    for ei, (code_snippet, _) in enumerate(EXT_SET, 1):
        o, po = answer_full_code(code_snippet, q)
        a, pa = answer_qxern(code_snippet, q)
        r, pr = answer_text_relay(code_snippet, q)
        oracle_list.append(o); qx_list.append(a); relay_list.append(r)
        ext_prefix["oracle"].append(po); ext_prefix["qxern"].append(pa); ext_prefix["text_relay"].append(pr)
        if ei % 5 == 0:
            print(f"  question {qi}/{len(EXT_QUESTIONS)}, example {ei}/{N_EXT} | elapsed {(time.perf_counter()-t_start)/60:.1f} min", flush=True)
    ext_sem[q] = {
        "baseline": semantic_sim([base_ans] * len(oracle_list), oracle_list),
        "qxern": semantic_sim(qx_list, oracle_list),
        "text_relay": semantic_sim(relay_list, oracle_list),
    }
    ext_ans[q] = (qx_list, relay_list, oracle_list)

# --- Results table ---
print("\nSemSim with the oracle (model B sees the full code as text):")
print(f"{'Question':62s} {'Baseline':>9s} {'Qxern':>9s} {'Relay':>9s} {'Qxern/Relay':>12s}")
for q, s in ext_sem.items():
    ratio = s["qxern"] / max(s["text_relay"], 1e-9)
    print(f"{q[:60]:62s} {s['baseline']:9.3f} {s['qxern']:9.3f} {s['text_relay']:9.3f} {ratio:12.2f}")
print(f"\nAverage decoder prefix: Qxern {np.mean(ext_prefix['qxern']):.1f} | "
      f"Relay {np.mean(ext_prefix['text_relay']):.1f} | Oracle (full code) {np.mean(ext_prefix['oracle']):.1f} positions")
print("How to read this: Qxern/Relay → the closer to 1.0 (or higher) on detailed questions,")
print("   the stronger the confirmation that relay won the main benchmark thanks to its head start,")
print("   not due to a fundamental superiority of the text channel.")

# --- Chart ---
labels = [q[:38] + ("…" if len(q) > 38 else "") for q in EXT_QUESTIONS]
x = np.arange(len(labels)); w = 0.27
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, [ext_sem[q]["baseline"] for q in EXT_QUESTIONS], w, label="Baseline (no code)", color="#f28e8e")
ax.bar(x,     [ext_sem[q]["qxern"] for q in EXT_QUESTIONS], w, label="Qxern (latent)", color="#6fce8f")
ax.bar(x + w, [ext_sem[q]["text_relay"] for q in EXT_QUESTIONS], w, label="Text Relay", color="#4f9dde")
ax.set_ylabel("SemSim with the oracle"); ax.set_title("Qxern v4 — detailed questions: closeness to the full-information answer")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=12, ha="right", fontsize=8); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "qxern_v5_ext_benchmark.png"), dpi=150, bbox_inches="tight"); plt.show()

# --- Qualitative comparison on the first example ---
print("\n" + "=" * 70)
print("Qualitative comparison (first test example):")
print("CODE:\n" + EXT_SET[0][0][:300])
for q in EXT_QUESTIONS:
    qx_list, relay_list, oracle_list = ext_ans[q]
    print(f"\nQ: {q}")
    print(f"  Qxern : {qx_list[0][:200]}")
    print(f"  Relay : {relay_list[0][:200]}")
    print(f"  Oracle: {oracle_list[0][:200]}")

## AST factual metrics: "measured, not guessed" — now in numbers

SemSim and the oracle measure *closeness*, not *factual correctness* — the main criticism of the previous evaluation. Here the ground truth comes straight from the **AST** of each real CodeSearchNet test function: parameter count, function name, whether it returns a value. Scoring is exact-match (digit or number-word for counts, the identifier for names, yes/no for returns) — cheap, objective, and on the real data distribution, with bootstrap 95% CIs.

This is intentionally done on real data first; fully synthetic benchmarks with perfect labels are a complement for later (they risk measuring the channel on a toy distribution of code).


In [ ]:
# 8. AST factual metrics: exact-match facts from the AST (real CodeSearchNet code)
import ast, re

WORD2NUM = {"zero": 0, "no": 0, "none": 0, "one": 1, "single": 1, "two": 2, "three": 3,
            "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10}

def ast_facts(code_text):
    """Ground truth straight from the AST: function name, parameter count, returns a value or not."""
    try:
        tree = ast.parse(code_text)
    except (SyntaxError, ValueError):
        return None
    fn = next((n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))), None)
    if fn is None:
        return None
    a = fn.args
    n_params = (len(getattr(a, "posonlyargs", [])) + len(a.args) + len(a.kwonlyargs)
                + (1 if a.vararg else 0) + (1 if a.kwarg else 0))
    has_return = any(isinstance(n, ast.Return) and n.value is not None for n in ast.walk(fn))
    return {"name": fn.name, "n_params": n_params, "has_return": has_return}

def _numbers_in(text):
    nums = set(int(v) for v in re.findall(r"\b\d+\b", text))
    low = text.lower()
    for w, v in WORD2NUM.items():
        if re.search(r"\b" + re.escape(w) + r"\b", low):
            nums.add(v)
    return nums

def score_param_count(answer, facts):
    return facts["n_params"] in _numbers_in(answer)

def score_name(answer, facts):
    return re.search(r"\b" + re.escape(facts["name"]) + r"\b", answer) is not None

def score_returns(answer, facts):
    low = answer.lower()
    says_yes = bool(re.search(r"\byes\b", low))
    says_no = bool(re.search(r"\bno\b", low)) and not says_yes
    if not (says_yes or says_no):
        return False  # unparseable = wrong; strict, but strict for every method equally
    return says_yes == facts["has_return"]

AST_TASKS = [
    ("How many parameters does this function take, and what are they?", score_param_count, "param count"),
    ("What is the name of this function?", score_name, "function name"),
    ("Does this function return a value? Answer yes or no.", score_returns, "returns value (yes/no)"),
]

N_AST = 30
ast_set = [(c, ast_facts(c)) for c, _ in TEST_SET]
ast_set = [(c, f) for c, f in ast_set if f is not None][:N_AST]
print(f"AST benchmark: {len(ast_set)} parseable test functions x {len(AST_TASKS)} facts x 4 methods")

AST_METHODS = {
    "baseline": lambda c, q: answer_baseline(question=q)[0],
    "qxern": lambda c, q: answer_qxern(c, q)[0],
    "text_relay": lambda c, q: answer_text_relay(c, q)[0],
    "oracle": lambda c, q: answer_full_code(c, q)[0],
}

ast_acc = {}
_t0 = time.perf_counter()
for q, scorer, label in AST_TASKS:
    for mname, fn in AST_METHODS.items():
        hits = []
        for code_text, facts in ast_set:
            ans = fn(code_text, q)
            hits.append(1.0 if scorer(ans, facts) else 0.0)
        ast_acc[(label, mname)] = np.asarray(hits)
    print(f"  done: {label} | elapsed {(time.perf_counter()-_t0)/60:.1f} min", flush=True)

print(f"\nExact-match accuracy vs AST ground truth (bootstrap 95% CI):")
print(f"{'Fact':<24} {'Baseline':>20} {'Qxern':>20} {'Relay':>20} {'Oracle':>20}")
for _, _, label in AST_TASKS:
    row = f"{label:<24}"
    for m in ["baseline", "qxern", "text_relay", "oracle"]:
        mm, lo, hi = bootstrap_ci(ast_acc[(label, m)])
        row += f" {mm:.2f} [{lo:.2f},{hi:.2f}]"
    print(row)

print("\nPaired bootstrap, Qxern vs Relay (per fact):")
for _, _, label in AST_TASKS:
    d, lo, hi = paired_bootstrap_diff(ast_acc[(label, "qxern")], ast_acc[(label, "text_relay")])
    verdict = "not established" if lo <= 0 <= hi else ("Qxern higher" if d > 0 else "Relay higher")
    print(f"  {label}: diff = {d:+.2f} [95% CI {lo:+.2f}, {hi:+.2f}] — {verdict}")

# --- Chart ---
fact_labels = [label for _, _, label in AST_TASKS]
mnames = ["baseline", "qxern", "text_relay", "oracle"]
mcolors = {"baseline": "#f28e8e", "qxern": "#6fce8f", "text_relay": "#4f9dde", "oracle": "#b18ae0"}
xx = np.arange(len(fact_labels)); w = 0.2
fig, ax = plt.subplots(figsize=(11, 5))
for j, m in enumerate(mnames):
    means, errs_lo, errs_hi = [], [], []
    for label in fact_labels:
        mm, lo, hi = bootstrap_ci(ast_acc[(label, m)])
        means.append(mm); errs_lo.append(mm - lo); errs_hi.append(hi - mm)
    ax.bar(xx + (j - 1.5) * w, means, w, yerr=[errs_lo, errs_hi], capsize=3, label=m, color=mcolors[m])
ax.set_ylabel("Exact-match accuracy vs AST"); ax.set_ylim(0, 1.05)
ax.set_title("Qxern v5 — AST factual accuracy (bootstrap 95% CI)")
ax.set_xticks(xx); ax.set_xticklabels(fact_labels); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "qxern_v5_ast_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()


## Channel capacity: quality vs number of latent tokens

The most valuable next chart per both reviews: train adapters with **8/16/32/64** latent tokens with exactly the same procedure and plot quality against channel width — the bottleneck becomes visible immediately. Each adapter is cached in `SAVE_DIR` (`qxern_adapter_<k>tok.pt`), so re-runs are cheap. This is the longest optional cell on a first run (3 extra trainings); on the RTX 5090 each training is far faster than the ~3 h it took on a T4. Set `RUN_CAPACITY_CURVE = False` in cell 1 to skip.


In [ ]:
# 9. Channel-capacity curve: quality vs number of latent tokens (8/16/32/64)

if RUN_CAPACITY_CURVE:
    curve = {}
    pre_ids_c, post_ids_c = chat_ids(EVAL_QUESTION)
    for k in LATENT_CURVE_SIZES:
        ad_k = adapter if k == NUM_LATENT_TOKENS else train_adapter(k)  # cached sizes are loaded, not retrained
        ad_k.eval()
        preds = []
        with torch.no_grad():
            for code_snippet, _ in TEST_SET:
                h, m = encode_one(code_snippet)
                latents = ad_k(h, m).to(model_b.dtype)
                inputs_embeds = torch.cat([embed_b(pre_ids_c), latents, embed_b(post_ids_c)], dim=1)
                ids = model_b.generate(inputs_embeds=inputs_embeds, max_new_tokens=GEN_TOKENS,
                                       pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
                preds.append(clean_answer(tokenizer_b.decode(ids[0], skip_special_tokens=True)) or "empty")
        sem = semantic_sim_per_example(preds, references)
        rl = rouge.compute(predictions=preds, references=references, use_aggregator=False)["rougeL"]
        curve[k] = {"semsim": bootstrap_ci(sem), "rougeL": bootstrap_ci(rl)}
        print(f"  {k:>3} tokens: SemSim {curve[k]['semsim'][0]:.4f} "
              f"[{curve[k]['semsim'][1]:.4f}, {curve[k]['semsim'][2]:.4f}] | "
              f"ROUGE-L {curve[k]['rougeL'][0]:.4f} "
              f"[{curve[k]['rougeL'][1]:.4f}, {curve[k]['rougeL'][2]:.4f}]", flush=True)
        if k != NUM_LATENT_TOKENS:
            del ad_k
        torch.cuda.empty_cache()

    ks = LATENT_CURVE_SIZES
    fig, ax1 = plt.subplots(figsize=(9, 5))
    for key, color, marker, lbl in [("semsim", "#2ecc71", "o", "SemSim vs docstring"),
                                    ("rougeL", "#3498db", "s", "ROUGE-L vs docstring")]:
        means = [curve[k][key][0] for k in ks]
        err = [[curve[k][key][0] - curve[k][key][1] for k in ks],
               [curve[k][key][2] - curve[k][key][0] for k in ks]]
        ax1.errorbar(ks, means, yerr=err, marker=marker, capsize=4, color=color, label=lbl)
    ax1.set_xscale("log", base=2)
    ax1.set_xticks(ks); ax1.set_xticklabels([str(k) for k in ks])
    ax1.set_xlabel("Latent tokens (channel width)"); ax1.set_ylabel("Score")
    ax1.set_title("Qxern v5 — channel capacity: quality vs number of latent tokens (bootstrap 95% CI)")
    ax1.grid(alpha=0.3); ax1.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "qxern_v5_capacity_curve.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("RUN_CAPACITY_CURVE = False — skipped.")


## Optional: LLM-judge on top of SemSim

SemSim is biased toward verbose answers (honestly noted in earlier versions). As a third signal, model A (the Coder) judges whether each main-benchmark answer is CORRECT or INCORRECT given the actual code. A 1.5B judge is imperfect — treat this as a *complement* to SemSim + AST metrics, not as ground truth; for a serious version, swap in a strong API model as the judge. Set `RUN_LLM_JUDGE = False` in cell 1 to skip.


In [ ]:
# 10. Optional: LLM-judge — factual verdicts on the main-benchmark answers

if RUN_LLM_JUDGE:
    JUDGE_FEWSHOT = (
        "Code:\ndef add(a, b):\n    return a + b\n"
        "Question: What does this function return?\n"
        "Answer: It returns the difference between a and b.\n"
        "Verdict: INCORRECT\n\n"
        "Code:\ndef add(a, b):\n    return a + b\n"
        "Question: What does this function return?\n"
        "Answer: The function returns the sum of its two arguments.\n"
        "Verdict: CORRECT\n\n")

    @torch.no_grad()
    def judge_answer(code_text, question, answer):
        prompt = JUDGE_FEWSHOT + f"Code:\n{code_text}\nQuestion: {question}\nAnswer: {answer.strip() or 'empty'}\nVerdict:"
        inp = tokenizer_a(prompt, return_tensors="pt", truncation=True,
                          max_length=MAX_LEN_CODE + 300).to(device)
        ids = model_a.generate(**inp, max_new_tokens=3, do_sample=False, num_beams=1,
                               pad_token_id=tokenizer_a.eos_token_id)
        verdict = tokenizer_a.decode(ids[0][inp["input_ids"].size(1):], skip_special_tokens=True).strip().upper()
        return verdict.startswith("CORRECT")

    judge_hits = {}
    for name in ["baseline", "qxern", "text_relay"]:
        judge_hits[name] = np.asarray([
            1.0 if judge_answer(code_text, EVAL_QUESTION, ans) else 0.0
            for (code_text, _), ans in zip(TEST_SET, results[name])
        ])
        mm, lo, hi = bootstrap_ci(judge_hits[name])
        print(f"LLM-judge 'CORRECT' rate — {name:<11}: {mm:.2f} [95% CI {lo:.2f}, {hi:.2f}]")

    d, lo, hi = paired_bootstrap_diff(judge_hits["qxern"], judge_hits["text_relay"])
    verdict = "not statistically established" if lo <= 0 <= hi else ("Qxern higher" if d > 0 else "Relay higher")
    print(f"\nQxern - Relay (judge): diff = {d:+.2f} [95% CI {lo:+.2f}, {hi:+.2f}] — {verdict}")
    print("Caveat: a 1.5B judge is noisy — use together with the AST metrics above, not instead of them.")
else:
    print("RUN_LLM_JUDGE = False — skipped.")


## 11 · Hybrid: latents (meaning) + AST sidecar (exact symbols)

We do not force the continuous channel to be a lossless codec. The Qxern packet becomes
`[semantic latent] + [signature / identifiers / literals]`.
The sidecar is extracted by a deterministic parser (`ast`, microseconds, can run in parallel with the encoder), costs **tens of tokens** and does not touch the trained adapter: returns and speed are protected by the construction itself. Payload levels are adaptive — we send exactly as much as the task needs.

In [ ]:
# 11. Symbolic sidecar: exact facts from the AST parser (sender side, NO LLM)
#
# Combined design decision (both plans agree):
#   - latents keep carrying the SEMANTICS ("what the code does") — this already works;
#   - a tiny symbolic sidecar carries the EXACT SYMBOLS (names, arity, flags, literals) verbatim;
#   - extraction is a cheap deterministic parser: microseconds, runs in parallel with the
#     encoder, so the ~6x speed advantage survives almost intact.
import ast

def sidecar_facts(code_text):
    """Exact, machine-checkable facts. Returns None if the code does not parse."""
    try:
        tree = ast.parse(code_text)
    except (SyntaxError, ValueError):
        return None
    fn = next((n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))), None)
    if fn is None:
        return None
    a = fn.args
    params = ([x.arg for x in getattr(a, "posonlyargs", [])] + [x.arg for x in a.args]
              + (["*" + a.vararg.arg] if a.vararg else [])
              + [x.arg for x in a.kwonlyargs] + (["**" + a.kwarg.arg] if a.kwarg else []))
    has_return = any(isinstance(n, ast.Return) and n.value is not None for n in ast.walk(fn))
    raises = []
    for n in ast.walk(fn):
        if isinstance(n, ast.Raise) and n.exc is not None:
            t = n.exc.func if isinstance(n.exc, ast.Call) else n.exc
            if isinstance(t, ast.Name) and t.id not in raises:
                raises.append(t.id)
    literals = []
    for n in ast.walk(fn):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float, str)) and n.value != "":
            v = repr(n.value)
            if len(v) <= 24 and v not in literals:
                literals.append(v)
    return {"name": fn.name, "params": params, "n_params": len(params),
            "has_return": has_return, "raises": raises[:3], "literals": literals[:5]}

# Adaptive payload levels (send only what the task needs):
#   S1 — signature only: function name + parameters
#   S2 — + behavioral flags: returns / raises            (default)
#   S3 — + critical literals
def render_sidecar(facts, level="S2"):
    if facts is None:
        return ""
    parts = [f"def {facts['name']}({', '.join(facts['params'])})",
             f"parameters: {facts['n_params']}"]
    if level in ("S2", "S3"):
        parts.append("returns a value: yes" if facts["has_return"] else "returns a value: no")
        if facts["raises"]:
            parts.append("raises: " + ", ".join(facts["raises"]))
    if level == "S3" and facts["literals"]:
        parts.append("literals: " + ", ".join(facts["literals"]))
    return " | ".join(parts)

def sidecar_cost(text):
    """(decoder tokens, utf-8 bytes) — the honest wire cost of the sidecar."""
    if not text:
        return 0, 0
    return len(tokenizer_b(text, add_special_tokens=False).input_ids), len(text.encode("utf-8"))

# Demo + payload statistics on the test set
_sc_tok, _sc_bytes, _sc_ok = [], [], 0
for _code, _ in TEST_SET:
    _f = sidecar_facts(_code)
    if _f is not None:
        _sc_ok += 1
        _t, _b = sidecar_cost(render_sidecar(_f, SIDECAR_LEVEL))
        _sc_tok.append(_t); _sc_bytes.append(_b)
print(f"Sidecar ({SIDECAR_LEVEL}) on the test set: parseable {_sc_ok}/{len(TEST_SET)} | "
      f"median {np.median(_sc_tok):.0f} tokens / {np.median(_sc_bytes):.0f} bytes | "
      f"p95 {np.percentile(_sc_tok, 95):.0f} tokens")
print(f"For scale: the latent packet is {NUM_LATENT_TOKENS} tokens x {DIM_B} dims x 2 bytes "
      f"~= {NUM_LATENT_TOKENS * DIM_B * 2 / 1024:.0f} KiB on the wire; "
      f"a median sidecar adds only ~{np.median(_sc_bytes):.0f} bytes on top.")
print("\nExample:")
print("CODE   :", TEST_SET[0][0][:160].replace(chr(10), " / "))
print("SIDECAR:", render_sidecar(sidecar_facts(TEST_SET[0][0]), SIDECAR_LEVEL) or "(unparseable)")


In [ ]:
# 12. Hybrid channel WITHOUT retraining: [latents] + [sidecar text] -> frozen decoder
# The minimal experiment both plans start with: if the frozen instruct-decoder can copy
# exact symbols out of a ~20-token sidecar, names/params should jump with ZERO training risk.
SIDECAR_PLACEHOLDER = "<<<SIDECAR>>>"
_hybrid_cache = {}

def _tok_ids(text):
    return tokenizer_b(text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)

def hybrid_chat_parts(question):
    """(pre, mid, post) token ids around the two insertion points (cached per question)."""
    if question not in _hybrid_cache:
        text = _chat_text(
            f"Here is the code (compressed):\n{LATENT_PLACEHOLDER}\n"
            f"Verified facts from a code parser: {SIDECAR_PLACEHOLDER}\n"
            f"Question: {question}")
        pre_text, rest = text.split(LATENT_PLACEHOLDER)
        mid_text, post_text = rest.split(SIDECAR_PLACEHOLDER)
        _hybrid_cache[question] = (_tok_ids(pre_text), _tok_ids(mid_text), _tok_ids(post_text))
    return _hybrid_cache[question]

@torch.no_grad()
def answer_latent(code, question=EVAL_QUESTION, ad=None, max_new_tokens=GEN_TOKENS):
    """Pure latent channel with a selectable adapter (v5 or struct)."""
    ad = adapter if ad is None else ad
    h, m = encode_one(code)
    latents = ad(h, m).to(model_b.dtype)
    pre_ids, post_ids = chat_ids(question)
    inputs_embeds = torch.cat([embed_b(pre_ids), latents, embed_b(post_ids)], dim=1)
    ids = model_b.generate(inputs_embeds=inputs_embeds, max_new_tokens=max_new_tokens,
                           pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
    return clean_answer(tokenizer_b.decode(ids[0], skip_special_tokens=True)), inputs_embeds.size(1)

@torch.no_grad()
def answer_hybrid(code, question=EVAL_QUESTION, ad=None, level=None, max_new_tokens=GEN_TOKENS):
    """Latents (semantics) + AST sidecar (exact symbols). End-to-end, incl. extraction time."""
    ad = adapter if ad is None else ad
    level = SIDECAR_LEVEL if level is None else level
    sc_text = render_sidecar(sidecar_facts(code), level)
    if not sc_text:  # unparseable code -> honest fallback to the pure latent channel
        return answer_latent(code, question, ad=ad, max_new_tokens=max_new_tokens)
    h, m = encode_one(code)
    latents = ad(h, m).to(model_b.dtype)
    pre, mid, post = hybrid_chat_parts(question)
    sc_ids = _tok_ids(sc_text)
    inputs_embeds = torch.cat([embed_b(pre), latents, embed_b(mid),
                               embed_b(sc_ids), embed_b(post)], dim=1)
    ids = model_b.generate(inputs_embeds=inputs_embeds, max_new_tokens=max_new_tokens,
                           pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
    return clean_answer(tokenizer_b.decode(ids[0], skip_special_tokens=True)), inputs_embeds.size(1)

# Smoke test: does the frozen decoder actually USE the sidecar?
for _code, _ in TEST_SET[:2]:
    print("-" * 70)
    print("SIDECAR :", render_sidecar(sidecar_facts(_code), SIDECAR_LEVEL))
    print("NAME Q  :", answer_hybrid(_code, "What is the name of this function?")[0][:140])
    print("PARAMS Q:", answer_hybrid(_code, "How many parameters does this function take, and what are they?")[0][:140])


## 13 · Contrastive probing: are the latents invariant to names?

The hypothesis behind `names = 0.00`: code differing **only in identifiers** collapses to (almost) a single point of latent space. If `cos(latent(code), latent(renamed)) ≈ 1`, identifiers physically never reach the decoder — no prompt or decoder can recover them, and the sidecar is not a "crutch" but an architectural necessity. This is a separately publishable result about channel geometry.

In [ ]:
# 13. Contrastive probing: rename identifiers only, compare latent packets
import ast

class _Renamer(ast.NodeTransformer):
    def __init__(self, mapping):
        self.mapping = mapping
    def visit_FunctionDef(self, node):
        self.generic_visit(node)
        node.name = self.mapping.get(node.name, node.name)
        return node
    visit_AsyncFunctionDef = visit_FunctionDef
    def visit_arg(self, node):
        node.arg = self.mapping.get(node.arg, node.arg)
        return node
    def visit_Name(self, node):
        node.id = self.mapping.get(node.id, node.id)
        return node

def rename_identifiers(code_text):
    """Rename the function and its parameters to opaque names; semantics is unchanged."""
    try:
        tree = ast.parse(code_text)
        fn = next((n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))), None)
        if fn is None:
            return None
        mapping = {fn.name: "qq_function"}
        a = fn.args
        for j, x in enumerate([*getattr(a, "posonlyargs", []), *a.args, *a.kwonlyargs]):
            mapping[x.arg] = f"qq_arg{j}"
        if a.vararg: mapping[a.vararg.arg] = "qq_va"
        if a.kwarg:  mapping[a.kwarg.arg] = "qq_kw"
        return ast.unparse(_Renamer(mapping).visit(tree))
    except Exception:
        return None

@torch.no_grad()
def latent_vec(code_text, ad=None):
    ad = adapter if ad is None else ad
    h, m = encode_one(code_text)
    z = ad(h, m).float().mean(dim=1)  # (1, DIM_B) — pooled latent packet
    return torch.nn.functional.normalize(z, dim=1).cpu()

_pairs = []
for _code, _ in TEST_SET:
    _r = rename_identifiers(_code)
    if _r is not None and _r != _code:
        _pairs.append((_code, _r))
_pairs = _pairs[:30]
print(f"Contrastive probing on {len(_pairs)} pairs (original vs identifier-renamed)...")

Z_orig = torch.cat([latent_vec(c) for c, _ in _pairs])
Z_ren  = torch.cat([latent_vec(r) for _, r in _pairs])
sim_same = (Z_orig * Z_ren).sum(1).numpy()                # same code, only ids renamed
_perm = np.roll(np.arange(len(_pairs)), 1)
sim_diff = (Z_orig * Z_orig[_perm]).sum(1).numpy()        # genuinely different functions

m_same, lo_s, hi_s = bootstrap_ci(sim_same)
m_diff, lo_d, hi_d = bootstrap_ci(sim_diff)
print(f"cos(original, renamed)  = {m_same:.4f} [{lo_s:.4f}, {hi_s:.4f}]  <- identifier sensitivity")
print(f"cos(original, other fn) = {m_diff:.4f} [{lo_d:.4f}, {hi_d:.4f}]  <- semantic separation")
if m_same > 0.98:
    print("VERDICT: latents are (almost) perfectly name-INVARIANT — identifiers are not encoded at all.")
    print("         The sidecar is architecturally necessary; retraining alone fights the channel's geometry.")
elif m_same > m_diff + 0.05:
    print("VERDICT: latents are largely name-invariant (renamed pairs are far closer than different code).")
    print("         Expect limited gains from retraining alone; the sidecar carries what latents drop.")
else:
    print("VERDICT: latents DO separate renamed variants — the bottleneck is the training signal,")
    print("         so the structured-retraining arm below may recover symbols even without a sidecar.")


## 14 · Training ablation: structured targets without a sidecar

"The channel learned exactly what it was rewarded for" → change the reward: AST-exact targets (name, arity, structured summary) **on top of 100% of the old supervision** (this is distill-old / anti-forgetting: all old examples stay in the dataset). Targets are free — the parser produces them, no new teacher generation is needed.

This branch answers the key ablation question: **how much comes from training, and how much from architecture (the sidecar)?** If names still do not transfer after changing the reward (and the probing above showed invariance), the problem is in the nature of the continuous channel, and the hybrid is the right separation of concerns. The `RUN_STRUCT_RETRAIN = False` flag fully disables this branch; the checkpoint is cached like the others.

In [ ]:
# 14. (optional ablation) Structured supervision WITHOUT a sidecar: "fix the reward"
# AST-exact targets (name, arity, structured summary) are ADDED on top of 100% of the old
# supervision — the old examples act as the distill-old / anti-forgetting term.
if RUN_STRUCT_RETRAIN:
    MAX_LEN_STRUCT = 112  # structured targets are slightly longer than plain descriptions

    def build_struct_examples():
        ex = list(EXAMPLES)  # keep ALL old supervision — protects SemSim and returns
        added = 0
        for i, (code, doc) in enumerate(TRAIN_DATA):
            f = sidecar_facts(code)
            if f is None:
                continue
            plist = ", ".join(f["params"]) if f["params"] else "none"
            ex.append((i, "What is the name of this function?",
                       f"The function is named {f['name']}."))
            ex.append((i, "How many parameters does this function take, and what are they?",
                       (f"It takes {f['n_params']} parameters: {plist}." if f["n_params"]
                        else "It takes no parameters.")))
            ex.append((i, "Give a structured summary of this code.",
                       f"Function: {f['name']}\nParameters ({f['n_params']}): {plist}\n"
                       f"Returns a value: {'yes' if f['has_return'] else 'no'}\nSummary: {doc}"))
            added += 3
        print(f"Structured examples: +{added} AST-exact targets on top of {len(EXAMPLES)} old ones.")
        return ex

    def train_adapter_v6(num_tokens, examples, tag, epochs=STRUCT_EPOCHS):
        """Same training procedure as v5's train_adapter, but with a custom example list
        and its own checkpoint name. Cached: delete the .pt file to retrain."""
        path = os.path.join(SAVE_DIR, f"qxern_adapter_{tag}_{num_tokens}tok.pt")
        ad = make_adapter(num_tokens)
        if os.path.exists(path):
            ad.load_state_dict(torch.load(path, map_location=device)); ad.eval()
            print(f"Found cached '{tag}' adapter: {path} — training SKIPPED.")
            return ad
        model_a.to("cpu"); torch.cuda.empty_cache()  # free VRAM for training
        t_ids_all = tokenizer_b([t for _, _, t in examples], return_tensors="pt",
                                padding="max_length", truncation=True,
                                max_length=MAX_LEN_STRUCT).input_ids
        ex_pair = torch.tensor([i for i, _, _ in examples])
        buckets = defaultdict(list)
        for ei, (_, q, _) in enumerate(examples):
            buckets[q].append(ei)
        optimizer = optim.AdamW(ad.parameters(), lr=LR, weight_decay=0.01)
        total_steps = epochs * math.ceil(len(examples) / BATCH_SIZE)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
        print(f"Training '{tag}' ({num_tokens} latent tokens): {len(examples)} examples, {epochs} epochs...")
        for epoch in range(1, epochs + 1):
            ad.train(); total_loss, n = 0.0, 0; t_epoch = time.perf_counter()
            batches = []
            for q_text, idxs in buckets.items():
                random.shuffle(idxs)
                for i in range(0, len(idxs), BATCH_SIZE):
                    batches.append((q_text, idxs[i:i + BATCH_SIZE]))
            random.shuffle(batches)
            optimizer.zero_grad()
            for q_text, ex_idx in batches:
                pair_idx = ex_pair[ex_idx]
                h = H_TRAIN[pair_idx].to(device).float()
                m = M_TRAIN[pair_idx].to(device)
                t_ids = t_ids_all[ex_idx].to(device)
                pre_ids, post_ids = chat_ids(q_text)
                pre_ids = pre_ids.expand(len(ex_idx), -1)
                post_ids = post_ids.expand(len(ex_idx), -1)
                latents = ad(h, m).to(model_b.dtype)
                inputs_embeds = torch.cat([embed_b(pre_ids), latents, embed_b(post_ids),
                                           embed_b(t_ids)[:, :-1, :]], dim=1)
                prefix_len = pre_ids.size(1) + latents.size(1) + post_ids.size(1)
                ignore = torch.full((len(ex_idx), prefix_len), -100, dtype=torch.long, device=device)
                labels = torch.cat([ignore, build_labels(t_ids)[:, :-1]], dim=1)
                loss = model_b(inputs_embeds=inputs_embeds, labels=labels).loss
                if torch.isnan(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(ad.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                total_loss += loss.item(); n += 1
                if n % 100 == 0:
                    el = time.perf_counter() - t_epoch
                    eta = el / n * (len(batches) - n)
                    print(f"  epoch {epoch}: {n}/{len(batches)} | loss {total_loss/n:.4f} | "
                          f"{el/60:.1f} min | ~{eta/60:.1f} min left", flush=True)
            print(f"Epoch {epoch}/{epochs} — CE Loss: {total_loss / max(n, 1):.4f}")
        torch.save(ad.state_dict(), path)
        ad.eval()
        model_a.to(device); model_a.eval(); torch.cuda.empty_cache()
        print(f"'{tag}' adapter saved -> {path}")
        return ad

    adapter_struct = train_adapter_v6(NUM_LATENT_TOKENS, build_struct_examples(), tag="struct")
else:
    adapter_struct = None
    print("RUN_STRUCT_RETRAIN = False — the training-ablation arm is skipped.")


## 15 · v6 main benchmark: full ablation + guard gates

Systems: `baseline` · `relay` (strengthened, permanent baseline) · `qxern_v5` (pure latents) · `hybrid` (architecture: latents + sidecar, no retraining) · `qxern_struct` (training: new reward, no sidecar) · `hybrid_struct` (both) · `oracle`.

We measure everything at once: AST facts (name/params/returns), SemSim on a held-out question, end-to-end latency (including AST extraction time), packet size — i.e. the **quality ↔ latency ↔ message size** curve, not a single "6x" number. At the end — hard PASS/FAIL gates ("did not make anything worse") and a paired bootstrap against relay. Metrics are NOT averaged into a single number: better names must not mask degraded behavioral facts.

In [ ]:
# 15. v6 MAIN benchmark: ablation with guard gates (AST facts + SemSim + latency + payload)
import json as _json6

try:  # defined in the extended-benchmark cell; re-define if that cell was skipped
    answer_full_code
except NameError:
    @torch.no_grad()
    def answer_full_code(code, question, max_new_tokens=GEN_TOKENS):
        full_ids = chat_prompt_ids(f"Here is the code:\n{code}\nQuestion: {question}")
        ids = model_b.generate(input_ids=full_ids, max_new_tokens=max_new_tokens,
                               pad_token_id=tokenizer_b.eos_token_id, **GEN_KWARGS)
        return (clean_answer(tokenizer_b.decode(ids[0][full_ids.size(1):], skip_special_tokens=True)),
                full_ids.size(1))

V6_SYSTEMS = {
    "baseline": lambda c, q: answer_baseline(question=q),
    "relay":    lambda c, q: answer_text_relay(c, q),
    "qxern_v5": lambda c, q: answer_latent(c, q, ad=adapter),
    "hybrid":   lambda c, q: answer_hybrid(c, q, ad=adapter),
}
if adapter_struct is not None:
    V6_SYSTEMS["qxern_struct"]  = lambda c, q: answer_latent(c, q, ad=adapter_struct)
    V6_SYSTEMS["hybrid_struct"] = lambda c, q: answer_hybrid(c, q, ad=adapter_struct)
V6_SYSTEMS["oracle"] = lambda c, q: answer_full_code(c, q)

FACT_LABELS = [label for _, _, label in AST_TASKS]  # from the v5 AST cell

# --- 1) AST factual accuracy: exact symbols + behavior ---
print(f"v6 facts: {len(ast_set)} functions x {len(AST_TASKS)} facts x {len(V6_SYSTEMS)} systems")
v6_ast = {}
_t0 = time.perf_counter()
for q, scorer, label in AST_TASKS:
    for sname, fn in V6_SYSTEMS.items():
        hits = [1.0 if scorer(fn(code_text, q)[0], facts) else 0.0 for code_text, facts in ast_set]
        v6_ast[(label, sname)] = np.asarray(hits)
    print(f"  facts done: {label} | {(time.perf_counter() - _t0)/60:.1f} min", flush=True)

# --- 2) SemSim + end-to-end latency on the held-out main question ---
print("Warm-up (excluded from timing)...")
for _w in range(2):
    for fn in V6_SYSTEMS.values():
        fn(TEST_SET[_w][0], EVAL_QUESTION)
_sync()

v6_answers = {s: [] for s in V6_SYSTEMS}
v6_lat = {s: [] for s in V6_SYSTEMS}
v6_prefix = {s: [] for s in V6_SYSTEMS}
_order_rng = random.Random(SEED + 7)
for code_snippet, _ in TEST_SET:
    order = list(V6_SYSTEMS); _order_rng.shuffle(order)
    for sname in order:
        _sync(); t0 = time.perf_counter()
        out, plen = V6_SYSTEMS[sname](code_snippet, EVAL_QUESTION)
        _sync()
        v6_lat[sname].append((time.perf_counter() - t0) * 1000)
        v6_answers[sname].append(out or "empty"); v6_prefix[sname].append(plen)

v6_sem = {s: semantic_sim_per_example(v6_answers[s], references) for s in V6_SYSTEMS}

# --- 3) Results table ---
def _ci_str(vals):
    mm, lo, hi = bootstrap_ci(vals)
    return f"{mm:.2f} [{lo:.2f},{hi:.2f}]"

print(f"\n{'System':<14} {'func name':>18} {'param count':>18} {'returns':>18} "
      f"{'SemSim':>18} {'p50 ms':>8} {'prefix':>7}")
print("-" * 110)
for s in V6_SYSTEMS:
    print(f"{s:<14} {_ci_str(v6_ast[('function name', s)]):>18} "
          f"{_ci_str(v6_ast[('param count', s)]):>18} "
          f"{_ci_str(v6_ast[('returns value (yes/no)', s)]):>18} "
          f"{_ci_str(v6_sem[s]):>18} {np.percentile(v6_lat[s], 50):>8.0f} "
          f"{np.mean(v6_prefix[s]):>7.1f}")

# --- 4) Paired bootstrap: each v6 candidate vs the strengthened relay ---
print("\nPaired bootstrap vs relay (CI containing 0 = difference NOT established):")
for cand in [s for s in ("hybrid", "qxern_struct", "hybrid_struct") if s in V6_SYSTEMS]:
    for label in FACT_LABELS:
        d, lo, hi = paired_bootstrap_diff(v6_ast[(label, cand)], v6_ast[(label, "relay")])
        verdict = "not established" if lo <= 0 <= hi else (f"{cand} higher" if d > 0 else "relay higher")
        print(f"  {cand:<14} {label:<24} diff = {d:+.2f} [{lo:+.2f}, {hi:+.2f}] — {verdict}")
    d, lo, hi = paired_bootstrap_diff(v6_sem[cand], v6_sem["relay"])
    verdict = "not established" if lo <= 0 <= hi else (f"{cand} higher" if d > 0 else "relay higher")
    print(f"  {cand:<14} {'SemSim':<24} diff = {d:+.4f} [{lo:+.4f}, {hi:+.4f}] — {verdict}")

# --- 5) GUARD GATES: hard "do no harm" criteria ---
print("\n=== GUARD GATES ===")
relay_p50 = float(np.percentile(v6_lat["relay"], 50))
base_sem = float(np.mean(v6_sem["qxern_v5"]))
v6_gates = {}
for cand in [s for s in ("hybrid", "hybrid_struct", "qxern_struct") if s in V6_SYSTEMS]:
    name_a = float(np.mean(v6_ast[("function name", cand)]))
    par_a  = float(np.mean(v6_ast[("param count", cand)]))
    ret_a  = float(np.mean(v6_ast[("returns value (yes/no)", cand)]))
    sem_a  = float(np.mean(v6_sem[cand]))
    speedup = relay_p50 / float(np.percentile(v6_lat[cand], 50))
    checks = [
        (f"returns >= {GUARDS['returns_min']:.2f}",            ret_a,   ret_a >= GUARDS["returns_min"]),
        (f"SemSim >= qxern_v5 - {GUARDS['semsim_drop_max']}",  sem_a,   sem_a >= base_sem - GUARDS["semsim_drop_max"]),
        (f"function name >= {GUARDS['name_min']:.2f}",         name_a,  name_a >= GUARDS["name_min"]),
        (f"param count >= {GUARDS['params_min']:.2f}",         par_a,   par_a >= GUARDS["params_min"]),
        (f"speedup vs relay >= {GUARDS['speedup_min']:.1f}x",  speedup, speedup >= GUARDS["speedup_min"]),
    ]
    ok = all(c[2] for c in checks)
    v6_gates[cand] = ok
    print(f"\n[{cand}] -> {'ALL GATES PASSED' if ok else 'GATES FAILED'}")
    for lbl, val, passed in checks:
        print(f"  {'PASS' if passed else 'FAIL'}  {lbl:<40} value = {val:.2f}")

# --- 6) Sidecar level ablation: how much payload does exactness need? ---
if RUN_SIDECAR_LEVEL_ABLATION:
    print("\nSidecar level ablation (hybrid on the frozen v5 adapter):")
    print(f"{'level':<6} {'func name':>10} {'params':>8} {'returns':>8} {'median tok':>11}")
    for lv in ("S1", "S2", "S3"):
        accs = {}
        for q, scorer, label in AST_TASKS:
            hits = [1.0 if scorer(answer_hybrid(c, q, level=lv)[0], f) else 0.0 for c, f in ast_set]
            accs[label] = float(np.mean(hits))
        toks = [sidecar_cost(render_sidecar(sidecar_facts(c), lv))[0] for c, _ in ast_set]
        print(f"{lv:<6} {accs['function name']:>10.2f} {accs['param count']:>8.2f} "
              f"{accs['returns value (yes/no)']:>8.2f} {np.median(toks):>11.0f}")

# --- 7) Persist everything for the README / paper ---
_summary = {
    "ast": {f"{label}|{s}": v6_ast[(label, s)].tolist() for label in FACT_LABELS for s in V6_SYSTEMS},
    "semsim": {s: np.asarray(v6_sem[s]).tolist() for s in V6_SYSTEMS},
    "latency_ms": {s: v6_lat[s] for s in V6_SYSTEMS},
    "prefix": {s: [float(x) for x in v6_prefix[s]] for s in V6_SYSTEMS},
    "gates": v6_gates,
    "config": {"sidecar_level": SIDECAR_LEVEL, "guards": GUARDS,
               "num_latent_tokens": NUM_LATENT_TOKENS, "struct_retrain": adapter_struct is not None},
}
with open(os.path.join(SAVE_DIR, "qxern_v6_results.json"), "w") as f:
    _json6.dump(_summary, f)
print(f"\nSaved -> {os.path.join(SAVE_DIR, 'qxern_v6_results.json')}")


In [ ]:
# 16. v6 visualization: facts by system + the quality <-> latency <-> payload trade-off
sys_names = list(V6_SYSTEMS)
_colors = {"baseline": "#f28e8e", "relay": "#4f9dde", "qxern_v5": "#6fce8f",
           "hybrid": "#2e8b57", "qxern_struct": "#e8b23f", "hybrid_struct": "#8a5cd6",
           "oracle": "#b0b0b0"}

# (a) factual accuracy by system
xx = np.arange(len(FACT_LABELS)); w = 0.8 / len(sys_names)
fig, ax = plt.subplots(figsize=(13, 5))
for j, s in enumerate(sys_names):
    means, lo_e, hi_e = [], [], []
    for label in FACT_LABELS:
        mm, lo, hi = bootstrap_ci(v6_ast[(label, s)])
        means.append(mm); lo_e.append(mm - lo); hi_e.append(hi - mm)
    ax.bar(xx + (j - (len(sys_names) - 1) / 2) * w, means, w, yerr=[lo_e, hi_e],
           capsize=2, label=s, color=_colors.get(s, "#777777"))
ax.set_ylabel("Exact-match accuracy vs AST"); ax.set_ylim(0, 1.05)
ax.set_title("Qxern v6 — AST factual accuracy: architecture (hybrid) vs training (struct) vs baselines")
ax.set_xticks(xx); ax.set_xticklabels(FACT_LABELS); ax.legend(ncol=4); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "qxern_v6_ast_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()

# (b) trade-off: latency (x) vs mean factual accuracy (y); marker size ~ decoder prefix (payload)
fig, ax = plt.subplots(figsize=(9, 6))
for s in sys_names:
    if s == "baseline":
        continue
    x = float(np.percentile(v6_lat[s], 50))
    y = float(np.mean([np.mean(v6_ast[(l, s)]) for l in FACT_LABELS]))
    sz = float(np.mean(v6_prefix[s]))
    ax.scatter(x, y, s=40 + 2 * sz, color=_colors.get(s, "#777777"), alpha=0.8, edgecolors="black")
    ax.annotate(f"{s}\n{sz:.0f} prefix pos", (x, y), textcoords="offset points",
                xytext=(8, 6), fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("End-to-end latency p50, ms (log scale)")
ax.set_ylabel("Mean AST factual accuracy (name / params / returns)")
ax.set_title("Qxern v6 — quality ↔ latency ↔ payload (marker size = decoder prefix)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "qxern_v6_tradeoff.png"), dpi=150, bbox_inches="tight")
plt.show()

# (c) SemSim with CIs — the "did not get worse" check at a glance
fig, ax = plt.subplots(figsize=(9, 4))
for j, s in enumerate(sys_names):
    mm, lo, hi = bootstrap_ci(v6_sem[s])
    ax.bar(j, mm, yerr=[[mm - lo], [hi - mm]], capsize=4, color=_colors.get(s, "#777777"))
ax.set_xticks(range(len(sys_names))); ax.set_xticklabels(sys_names, rotation=15)
ax.set_ylabel("SemSim vs reference docstring")
ax.set_title("Qxern v6 — semantic quality on the held-out question (bootstrap 95% CI)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "qxern_v6_semsim.png"), dpi=150, bbox_inches="tight")
plt.show()


## 17 · Adaptive router: pay for exactness only when it is needed

The mode is chosen per question type: semantics/behavior → pure latents (fastest, returns is already 0.90); exact symbols → latents + sidecar; verbatim reproduction → rare fallback to relay. The evaluation below reuses already-measured results — no new generations are needed.

In [ ]:
# 17. Adaptive router: latent-only / latent+sidecar / relay, chosen per question
import re as _re

_VERBATIM_RE = _re.compile(r"verbatim|word.for.word|reproduce.*(code|source)|exact (source|code)|full (source|code)", _re.I)
_SYMBOL_RE   = _re.compile(r"name|named|signature|parameter|argument|identifier|literal|constant|exactly|spell", _re.I)

def route(question):
    if _VERBATIM_RE.search(question):
        return "relay"       # rare fallback: latents were never meant to be a lossless codec
    if _SYMBOL_RE.search(question):
        return "hybrid"      # exact symbols required -> pay the ~20-token sidecar
    return "qxern_v5"        # semantics/behavior is enough -> fastest channel

print("Routing decisions:")
_demo_qs = [q for q, _, _ in AST_TASKS] + [EVAL_QUESTION, "Reproduce the exact code of this function."]
for q in _demo_qs:
    print(f"  {route(q):>9s}  <-  {q}")

print("\nRouted quality on measured results (no new generations):")
print(f"{'task':<26} {'routed to':<10} {'router':>7} {'relay':>7} {'qxern_v5':>9} {'hybrid':>7}")
_routed_lat = []
for q, _, label in AST_TASKS:
    r = route(q)
    _routed_lat.append(float(np.percentile(v6_lat[r], 50)))
    print(f"{label:<26} {r:<10} {np.mean(v6_ast[(label, r)]):>7.2f} "
          f"{np.mean(v6_ast[(label, 'relay')]):>7.2f} "
          f"{np.mean(v6_ast[(label, 'qxern_v5')]):>9.2f} "
          f"{np.mean(v6_ast[(label, 'hybrid')]):>7.2f}")
_r_main = route(EVAL_QUESTION)
_routed_lat.append(float(np.percentile(v6_lat[_r_main], 50)))
print(f"{'main question (SemSim)':<26} {_r_main:<10} {np.mean(v6_sem[_r_main]):>7.2f} "
      f"{np.mean(v6_sem['relay']):>7.2f} {np.mean(v6_sem['qxern_v5']):>9.2f} "
      f"{np.mean(v6_sem['hybrid']):>7.2f}")

print(f"\nMean p50 latency over this question mix:")
print(f"  router        : {np.mean(_routed_lat):.0f} ms")
print(f"  always-hybrid : {np.percentile(v6_lat['hybrid'], 50):.0f} ms")
print(f"  always-relay  : {np.percentile(v6_lat['relay'], 50):.0f} ms")
print("\nReading: the router should match hybrid on symbol questions and match pure latents")
print("on semantic/behavioral ones — exactness where needed, speed everywhere else.")


## How to phrase the claim now

**Before (v4):** "latents are better and faster than text" — did not survive an honest relay.

**Now (v6):** *Latents are a fast semantic channel; a compact AST sidecar restores exact symbols only when required.* Qxern is not a "magic replacement for text" but an adaptive code-transmission protocol: latents for meaning, symbols for exactness, relay as a rare fallback.

**To lock into the README permanently:**
- the strengthened relay as a permanent baseline + guard gates (returns, SemSim, latency) — so the "pretty v4 numbers" never come back;
- the **quality ↔ latency ↔ packet size** curve (`qxern_v6_tradeoff.png`), not a single "6x" number: even if the hybrid gives 4–5x instead of 6x, that is a strong result given the sharp recovery of exactness;
- contrastive probing as the explanation of names = 0.00 (channel geometry, not a bug) — a separately publishable result;
- the "training vs architecture" ablation (`qxern_struct` vs `hybrid`): it shows what exactly restores the symbols.

**What NOT to do (locked in by both plans):**
- do not grow the latent width further (the curve is flat);
- do not retrain everything end-to-end from scratch — returns can be lost;
- do not push the whole code through the sidecar — that is relay in disguise;
- do not average metrics into one number — better names must not mask degraded behavior facts;
- do not claim "lossless quality" until latency and sidecar size are measured.

**Next steps (outside this notebook):**
1. If the zero-shot hybrid uses the sidecar imperfectly — a small fusion adapter (copy/pointer mechanism) trained with the semantic path frozen (Stage A of the plan).
2. VQ / discrete bottleneck — a "pure" latent solution for symbols if the hybrid is "inelegant" for a paper: continuous latents interpolate, symbols do not.
3. A confidence gate based on decoder entropy for automatic relay fallback (the router currently switches by question type).